### Setup

In [1]:
# Library
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"   
os.environ["CUDA_VISIBLE_DEVICES"]="1"
import torch
import random
import numpy as np
from datetime import datetime
import json
import shutil
from datasets import Dataset, DatasetDict
from PIL import Image
from torchvision import transforms
from torchmetrics.image.fid import FrechetInceptionDistance
from transformers import CLIPTokenizer
from huggingface_hub import login
from diffusers import DDPMScheduler, StableDiffusionPipeline, StableDiffusionXLPipeline, AutoPipelineForText2Image
from metric import *

from safetensors.torch import load_file
from transformers import CLIPVisionModelWithProjection
from diffusers.models import UNet2DConditionModel
from diffusers.models.attention_processor import LoRAAttnProcessor, LoRAAttnAddedKVProcessor
from diffusers import KandinskyV22Pipeline, KandinskyV22PriorPipeline

# GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device : ', device)   

# CONFIG
CURRENT_TIME = datetime.now().strftime("%Y%m%d_%H%M%S")
MASTER_SEED = 42
IMG_SIZE = 512
BATCH_SIZE = 4
TRAIN_DATA_SIZE = 2024
TEST_DATA_SIZE = 506

# PATH
CONFIG_PATH = '../config.json'
with open(CONFIG_PATH,'r') as f:
    config = json.load(f)
## DATA
TEST_IMAGE_FOLDER = config.get("TEST_IMAGE_FOLDER")   
TEST_LABEL_FOLDER = config.get("TEST_LABEL_EXPAND_FOLDER") 
TEST_IMAGE_FILE = sorted([file for file in os.listdir(TEST_IMAGE_FOLDER) if file.endswith(('.jpg', '.jpeg', '.png'))], key=lambda x: int(x.split('.')[0]))[:TEST_DATA_SIZE]
TEST_LABEL_FILE = sorted([file for file in os.listdir(TEST_LABEL_FOLDER) if file.endswith(('.json'))], key=lambda x: int(x.split('.')[0]))[:TEST_DATA_SIZE]

## DATA for nike, zara
# TEST_IMAGE_FOLDER = config.get("NIKE_IMAGE_FOLDER") 
# TEST_LABEL_FOLDER = config.get("NIKE_LABEL_FOLDER") 
# TEST_IMAGE_FOLDER = config.get("ZARA_IMAGE_FOLDER") 
# TEST_LABEL_FOLDER = config.get("ZARA_LABEL_FOLDER") 
# TEST_IMAGE_FILE = sorted([file for file in os.listdir(TEST_IMAGE_FOLDER) if file.endswith(('.jpg', '.jpeg', '.png'))], key=lambda x: int(x.split('.')[0][4:]))[TRAIN_DATA_SIZE:]
# TEST_LABEL_FILE = sorted([file for file in os.listdir(TEST_LABEL_FOLDER) if file.endswith(('.json'))], key=lambda x: int(x.split('.')[0][4:]))[TRAIN_DATA_SIZE:]

# HUGGINGFACE
HUGGING_FACE_TOKEN = config.get("HUGGING_FACE_TOKEN")
login(HUGGING_FACE_TOKEN)

/home/gayeon38/miniconda3/envs/figma_kandinksy/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/gayeon38/miniconda3/envs/figma_kandinksy/lib/python3.10/site-packages/diffusers-0.24.0.dev0-py3.10.egg/diffusers/utils/outputs.py:63: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
2025-06-20 18:13:58.590502: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750410838.601411  962261 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:

Instructions for updating:
non-resource variables are not supported in the long term
device :  cuda


### Preprocess Data

- Model : SD

In [ ]:
# Config
PRE_TRAINED_MODEL_NAME="stablediffusionapi/deliberate-v2"
SAVE_WEIGHTS_PATH = 'FIGMA-main/Experiment/model_weights/SD_label_expand/FIGMA_weights_20250618_140200'

# load SD original model 
pipe = StableDiffusionPipeline.from_pretrained(PRE_TRAINED_MODEL_NAME, torch_dtype=torch.float16).to(device)

# load fine-tunined model weights and set
# pipe.load_lora_weights(SAVE_WEIGHTS_PATH, safe_serialization=True)

tokenizer = pipe.tokenizer
text_encoder = pipe.text_encoder.to(torch.float16).to(device)  
vae = pipe.vae.to(torch.float16).to(device)  
unet = pipe.unet.to(torch.float16).to(device)  

noise_scheduler = DDPMScheduler.from_pretrained(
    PRE_TRAINED_MODEL_NAME,
    subfolder="scheduler"
)
if hasattr(noise_scheduler, "alphas_cumprod"):
    noise_scheduler.alphas_cumprod = noise_scheduler.alphas_cumprod.to(device)

- Model : SDXL

In [ ]:
# Config
PRE_TRAINED_MODEL_NAME="stabilityai/stable-diffusion-xl-base-1.0"
SAVE_WEIGHTS_PATH = 'FIGMA-main/Experiment/model_weights/SDXL_label_expand/pytorch_lora_weights.safetensors'

# load SDXL original model  
pipe = StableDiffusionXLPipeline.from_pretrained(PRE_TRAINED_MODEL_NAME, torch_dtype=torch.float16).to(device)

# load fine-tunined model weights and set
# pipe.load_lora_weights(SAVE_WEIGHTS_PATH)

tokenizer = pipe.tokenizer
text_encoder = pipe.text_encoder.to(torch.float16).to(device)  
vae = pipe.vae.to(torch.float16).to(device)  
unet = pipe.unet.to(torch.float16).to(device)  

noise_scheduler = DDPMScheduler.from_pretrained(
    PRE_TRAINED_MODEL_NAME,
    subfolder="scheduler"
)
if hasattr(noise_scheduler, "alphas_cumprod"):
    noise_scheduler.alphas_cumprod = noise_scheduler.alphas_cumprod.to(device)

Loading pipeline components...: 100%|██████████| 7/7 [00:01<00:00,  3.55it/s]


- Model : KANDINSKY

In [2]:
# Config
PRE_TRAINED_DECODER_MODEL_NAME = 'kandinsky-community/kandinsky-2-2-decoder'
PRE_TRAINED_PRIOR_MODEL_NAME = 'kandinsky-community/kandinsky-2-2-prior'
SAVE_DECODER_WEIGHTS_PATH = '../Experiment/model_weights/KANDINSKY_decoder_label_exapnd/model.safetensors'
SAVE_PRIOR_WEIGHTS_PATH = '../Experiment/model_weights/KANDINSKY_prior_label_exapnd/model.safetensors'

# load KANDINSKY original model  
image_encoder = CLIPVisionModelWithProjection.from_pretrained('kandinsky-community/kandinsky-2-2-prior', subfolder='image_encoder').to(torch.float16).to(device)
unet = UNet2DConditionModel.from_pretrained('kandinsky-community/kandinsky-2-2-decoder', subfolder='unet').to(torch.float16).to(device)
tokenizer = CLIPTokenizer.from_pretrained('kandinsky-community/kandinsky-2-2-prior',subfolder='tokenizer')
prior = KandinskyV22PriorPipeline.from_pretrained('kandinsky-community/kandinsky-2-2-prior', image_encoder=image_encoder, torch_dtype=torch.float16).to(device)
decoder = KandinskyV22Pipeline.from_pretrained('kandinsky-community/kandinsky-2-2-decoder', unet=unet, torch_dtype=torch.float16).to(device)

# load fine-tunined model weights and set
tune_decoder = load_file(SAVE_DECODER_WEIGHTS_PATH)
lora_attn_procs = {}
for name in decoder.unet.attn_processors.keys():
    cross_attention_dim = None if name.endswith("attn1.processor") else decoder.unet.config.cross_attention_dim
    if name.startswith("mid_block"):
        hidden_size = decoder.unet.config.block_out_channels[-1]
    elif name.startswith("up_blocks"):
        block_id = int(name[len("up_blocks.")])
        hidden_size = list(reversed(decoder.unet.config.block_out_channels))[block_id]
    elif name.startswith("down_blocks"):
        block_id = int(name[len("down_blocks.")])
        hidden_size = decoder.unet.config.block_out_channels[block_id]
    lora_attn_procs[name] = LoRAAttnAddedKVProcessor(
            hidden_size=hidden_size,
            cross_attention_dim=cross_attention_dim,
            rank=4,
    ).to(device)
decoder.unet.set_attn_processor(lora_attn_procs)
decoder.unet.load_state_dict(tune_decoder, strict=False)

tune_encoder = load_file(SAVE_PRIOR_WEIGHTS_PATH)
lora_attn_procs = {}
for name in prior.prior.attn_processors.keys():
    lora_attn_procs[name] = LoRAAttnProcessor(hidden_size=2048).to(device)
prior.prior.set_attn_processor(lora_attn_procs)
prior.prior.load_state_dict(tune_encoder, strict=False)
None

Loading pipeline components...: 100%|██████████| 3/3 [00:00<00:00, 34.50it/s]


- Process Data with Model

In [3]:
from torchvision import transforms as T
# Function to tokenize the text column in the given data sample and return token ID tensors.
def tokenize_captions(examples, caption_column='text', is_train=True):
    captions = []
    for caption in examples[caption_column]:
        if isinstance(caption, str):
            captions.append(caption)
        elif isinstance(caption, (list, np.ndarray)):
            # Take a random caption if training, otherwise take the first one
            captions.append(random.choice(caption) if is_train else caption[0])
        else:
            raise ValueError(
                f"Caption column `{caption_column}` should contain either strings or lists of strings."
            )
    
    inputs = tokenizer(
        captions,
        max_length=tokenizer.model_max_length,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )
    return inputs.input_ids

# Transformation pipeline for preprocessing image data for training
image_transforms = T.Compose([
    T.Resize(IMG_SIZE, interpolation=T.InterpolationMode.BILINEAR),
    T.CenterCrop(IMG_SIZE),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize([0.5], [0.5])  # Convert (0,1) -> (-1,1)
])

# Function to preprocess images and tokenize text column
def preprocess_data(examples, image_column='image'):
    images = [image.convert("RGB") for image in examples[image_column]]
    examples["pixel_values"] = [image_transforms(image) for image in images]
    examples["input_ids"] = tokenize_captions(examples)
    return examples

# Function to batch images and token IDs
def collate_fn(examples):
    pixel_values = torch.stack([example["pixel_values"] for example in examples])
    pixel_values = pixel_values.to(memory_format=torch.contiguous_format).float()

    input_ids = torch.stack([example["input_ids"] for example in examples])

    return {
        "pixel_values": pixel_values,
        "input_ids": input_ids
    }

In [4]:
data = []
for image_file, label_file in zip(TEST_IMAGE_FILE, TEST_LABEL_FILE):
    image_path = os.path.join(TEST_IMAGE_FOLDER, image_file)
    label_path = os.path.join(TEST_LABEL_FOLDER, label_file)
    with open(image_path, 'rb') as image:
        image_data = Image.open(image)
        image_data = image_data.convert('RGB')
    with open(label_path, 'r') as label:
        label_data = json.load(label)
    data.append({
        'image':image_data,
        'text':label_data
    })
    
# Converting to Dataset
if TEST_IMAGE_FOLDER[23:] == "Nike/" or TEST_IMAGE_FOLDER[23:] == "Zara/":
    dataset = Dataset.from_dict({
        'image': [item['image'] for item in data],
        'text': [item['text']['Summary'] for item in data]
    })
else:
    dataset = Dataset.from_dict({
        'image': [item['image'] for item in data],
        'text': [item['text'] for item in data]
    })
# Create Test DatasetDict and Apply Preprocessing
test_dataset = DatasetDict({'test': dataset})
test_dataset = test_dataset.with_transform(preprocess_data)
test_dataloader = torch.utils.data.DataLoader(
    test_dataset['test'],
    shuffle=False,
    collate_fn=collate_fn,
    batch_size=BATCH_SIZE
)
print(test_dataset)

DatasetDict({
    test: Dataset({
        features: ['image', 'text'],
        num_rows: 506
    })
})


### Test Model

In [5]:
def calculate_fid(test_dataset, test_label_file, tmp_folder, pipe):
    from torchvision import transforms as T
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    fid_metric = FrechetInceptionDistance(feature=64).to(device)
    fid_metric.reset()
    
    # Use transform to convert PIL images to uint8 tensors
    transform = T.PILToTensor()
    
    # Create a temporary folder (for storing created images)
    os.makedirs(tmp_folder, exist_ok=True)
    for idx, filename in tqdm(enumerate(test_label_file)):
        prompt = test_dataset['test'][idx]['text'][16:]
        generated_image = pipe(prompt).images[0]
        # Save generate image   
        filename = filename.split('.')[0]
        generated_image.save(f'{tmp_folder}/{filename}.jpg')
    
        # real image
        real_image = test_dataset['test'][idx]['image']
        
        fake_tensor = transform(generated_image).unsqueeze(0).to(device)
        real_tensor = transform(real_image).unsqueeze(0).to(device)
        
        fid_metric.update(real_tensor, real=True)
        fid_metric.update(fake_tensor, real=False)
    
    fid_score = fid_metric.compute().item()
    return fid_score

def calculate_fid_kandinsky(test_dataset, test_label_file, tmp_folder, prior, decoder):
    from torchvision import transforms as T
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    fid_metric = FrechetInceptionDistance(feature=64).to(device)
    fid_metric.reset()    
    os.makedirs(tmp_folder, exist_ok=True)
    # Use transform to convert PIL images to uint8 tensors
    transform = T.PILToTensor()
    
    for idx, filename in tqdm(enumerate(test_label_file)):
        prompt = test_dataset['test'][idx]['text'][16:]
        negative_prompt = "low quality, low resolution, missing fingers, include head"
        
        # get embeddings from prior
        img_emb = prior(prompt=prompt, num_inference_steps=25, num_images_per_prompt=1)
        negative_emb = prior(prompt=negative_prompt, num_inference_steps=25, num_images_per_prompt=1)

        # generate image
        output = decoder(
            image_embeds=img_emb.image_embeds,
            negative_image_embeds=negative_emb.image_embeds,
            num_inference_steps=50,
            height=512,
            width=512
        )
        generated_image = output.images[0]

        # Save generated image
        filename = filename.split('.')[0]
        generated_image.save(f'{tmp_folder}/{filename}.jpg')

        # real image
        real_image = test_dataset['test'][idx]['image']

        # transform to tensor
        fake_tensor = transform(generated_image).unsqueeze(0).to(device)
        real_tensor = transform(real_image).unsqueeze(0).to(device)

        # update FID
        fid_metric.update(real_tensor, real=True)
        fid_metric.update(fake_tensor, real=False)
    
    fid_score = fid_metric.compute().item()
    return fid_score

- Model : SD

In [ ]:
# Compute FID score
GENERATE_IMAGE_FOLDER = '../Data/Total/tmp'
fid_score = calculate_fid(test_dataset, TEST_LABEL_FILE, GENERATE_IMAGE_FOLDER, pipe)

# data : train
GENERATE_IMAGE_FILE = sorted([file for file in os.listdir(GENERATE_IMAGE_FOLDER)], key=lambda x: int(x.split('.')[0]))
# data : nike or zara
# GENERATE_IMAGE_FILE = sorted([file for file in os.listdir(GENERATE_IMAGE_FOLDER)], key=lambda x: int(x.split('.')[0][4:]))

# Initialize TensorFlow session and evaluator
config = tf.ConfigProto(allow_soft_placement=True)
config.gpu_options.allow_growth = True
sess = tf.Session(config=config)
evaluator = Evaluator(sess)
evaluator.warmup()

# Prepare batch generator (original images)
seed_batches = batch_generator(TEST_IMAGE_FILE, TEST_IMAGE_FOLDER, batch_size=64)
# Prepare batch generator (generated images)
augment_batches = batch_generator(GENERATE_IMAGE_FILE, GENERATE_IMAGE_FOLDER, batch_size=64)

# Extract activations for original images (pool_3 features)
seed_activations = evaluator.compute_activations(seed_batches) 
augment_activations = evaluator.compute_activations(augment_batches)

# Compute precision and recall
precision, recall = evaluator.compute_prec_recall(seed_activations[0], augment_activations[0])
f1 = 2 * precision * recall / (precision + recall) 
# Remove folder if it exists
shutil.rmtree(GENERATE_IMAGE_FOLDER)

print(f'FID Score: {fid_score}, Precision: {precision}, Recall: {recall}, F1: {f1}')

- Model : SDXL

In [ ]:
# Compute FID score
GENERATE_IMAGE_FOLDER = '../Data/Total/tmp'
fid_score = calculate_fid(test_dataset, TEST_LABEL_FILE, GENERATE_IMAGE_FOLDER, pipe)

# data : train
GENERATE_IMAGE_FILE = sorted([file for file in os.listdir(GENERATE_IMAGE_FOLDER)], key=lambda x: int(x.split('.')[0]))
# data : nike or zara
# GENERATE_IMAGE_FILE = sorted([file for file in os.listdir(GENERATE_IMAGE_FOLDER)], key=lambda x: int(x.split('.')[0][4:]))

# Initialize TensorFlow session and evaluator
config = tf.ConfigProto(allow_soft_placement=True)
config.gpu_options.allow_growth = True
sess = tf.Session(config=config)
evaluator = Evaluator(sess)
evaluator.warmup()

# Prepare batch generator (original images)
seed_batches = batch_generator(TEST_IMAGE_FILE, TEST_IMAGE_FOLDER, batch_size=64)
# Prepare batch generator (generated images)
augment_batches = batch_generator(GENERATE_IMAGE_FILE, GENERATE_IMAGE_FOLDER, batch_size=64)

# Extract activations for original images (pool_3 features)
seed_activations = evaluator.compute_activations(seed_batches) 
augment_activations = evaluator.compute_activations(augment_batches)

# Compute precision and recall
precision, recall = evaluator.compute_prec_recall(seed_activations[0], augment_activations[0])
f1 = 2 * precision * recall / (precision + recall) 
# Remove folder if it exists
shutil.rmtree(GENERATE_IMAGE_FOLDER)

print(f'FID Score: {fid_score}, Precision: {precision}, Recall: {recall}, F1: {f1}')

- Model : KANDISKY

In [ ]:
# Compute FID score
GENERATE_IMAGE_FOLDER = '../Data/Total/tmp'
fid_score = calculate_fid_kandinsky(test_dataset, TEST_LABEL_FILE, GENERATE_IMAGE_FOLDER, prior, decoder)

# data : train
GENERATE_IMAGE_FILE = sorted([file for file in os.listdir(GENERATE_IMAGE_FOLDER)], key=lambda x: int(x.split('.')[0]))
# data : nike or zara
# GENERATE_IMAGE_FILE = sorted([file for file in os.listdir(GENERATE_IMAGE_FOLDER)], key=lambda x: int(x.split('.')[0][4:]))

# Initialize TensorFlow session and evaluator
config = tf.ConfigProto(allow_soft_placement=True)
config.gpu_options.allow_growth = True
sess = tf.Session(config=config)
evaluator = Evaluator(sess)
evaluator.warmup()

# Prepare batch generator (original images)
seed_batches = batch_generator(TEST_IMAGE_FILE, TEST_IMAGE_FOLDER, batch_size=64)
# Prepare batch generator (generated images)
augment_batches = batch_generator(GENERATE_IMAGE_FILE, GENERATE_IMAGE_FOLDER, batch_size=64)

# Extract activations for original images (pool_3 features)
seed_activations = evaluator.compute_activations(seed_batches) 
augment_activations = evaluator.compute_activations(augment_batches)

# Compute precision and recall
precision, recall = evaluator.compute_prec_recall(seed_activations[0], augment_activations[0])
f1 = 2 * precision * recall / (precision + recall) 
# Remove folder if it exists
shutil.rmtree(GENERATE_IMAGE_FOLDER)

print(f'FID Score: {fid_score}, Precision: {precision}, Recall: {recall}, F1: {f1}')